# General Biochemistry Study Buddy Agentic RAG with LLaVA

This notebook runs an **agentic RAG** study buddy chatbot for biochemistry entirely in Google Colab,  
and includes a built-in **RAG evaluation suite** (Faithfulness, Relevance, Compliance).

**What it does:**
- Loads your course materials (PDFs, DOCX, PPTX) from Google Drive
- Indexes them into a searchable vector database (ChromaDB)
- Uses **LLaVA** (a vision + language AI) via Ollama as the brain
- A **LangGraph agent** decides whether to search your notes or just chat
- A **Gradio chat UI** lets you ask questions and see traces, sources, and the answer
- An **Sentence-transformer** scores the pipeline on Faithfulness, Relevance, and Compliance

**Requirements:** Use a **GPU runtime** Runtime menu  Change runtime type  **T4 GPU**

---
| Section | What you get |
|---|---|
| Steps 1- 7 | Full agentic RAG chat system + Gradio UI |
| Steps 8- 13 | RAG evaluation suite with scores and charts |

## Step 1 Install Python packages

In [ ]:
!pip install -q \
    langchain langchain-core langchain-community langchain-ollama \
    langchain-chroma langchain-text-splitters langgraph \
    chromadb sentence-transformers \
    pypdf python-docx docx2txt python-pptx \
    PyMuPDF \
    gradio pydantic \
    pandas matplotlib

# LibreOffice is needed to render PPTX slides as images
!apt-get install -y -qq libreoffice > /dev/null 2>&1

print("All packages installed (including PyMuPDF + LibreOffice + pandas + matplotlib).")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1

## Step 2 Install Ollama and pull the LLaVA model

This downloads Ollama, starts the server in the background, then pulls **LLaVA 7b** (works on a free-tier T4 GPU) and the **nomic-embed-text** embedding model.



In [ ]:
import subprocess, time, requests, os

ollama_path = "/usr/local/bin/ollama"

# --- Install prerequisites ---
print("Installing prerequisites (zstd)...")
subprocess.run("apt-get install -y zstd", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# --- Cleanup previous broken installations ---
if os.path.exists(ollama_path):
    print("Removing old corrupted Ollama binary...")
    os.remove(ollama_path)

# --- Install Ollama ---
print("Installing Ollama...")
result = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print("Install script failed.")
    print("stdout:", result.stdout)
    print("stderr:", result.stderr)

# Verify the downloaded file
print("Verifying Ollama binary...")
if not os.path.exists(ollama_path):
    raise FileNotFoundError(f"Ollama binary not found at {ollama_path} after download.")

# --- Start Ollama server in background ---
print("Starting Ollama server...")
subprocess.Popen([ollama_path, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for server to be ready
for i in range(30):
    try:
        r = requests.get("http://localhost:11434")
        if r.status_code == 200:
            print("Ollama server is running.")
            break
    except:
        pass
    time.sleep(2)
else:
    raise RuntimeError("Ollama server did not start in time.")

# --- Pull models ---
LLAVA_MODEL = "llava:7b"
EMBED_MODEL_PULL = "nomic-embed-text"

print(f"\nPulling {LLAVA_MODEL} (this takes a few minutes)...")
subprocess.run([ollama_path, "pull", LLAVA_MODEL], check=True)

print(f"\nPulling {EMBED_MODEL_PULL}...")
subprocess.run([ollama_path, "pull", EMBED_MODEL_PULL], check=True)

# Verify
print("\n--- Installed models ---")
subprocess.run([ollama_path, "list"])

Installing prerequisites (zstd)...
Removing old corrupted Ollama binary...
Installing Ollama...
Verifying Ollama binary...
Starting Ollama server...
Ollama server is running.

Pulling llava:7b (this takes a few minutes)...

Pulling nomic-embed-text...

--- Installed models ---


CompletedProcess(args=['/usr/local/bin/ollama', 'list'], returncode=0)

## Step 3 Mount Google Drive and set your course materials folder

1. Run this cell it will ask you to log in to Google
2. After mounting, set `COURSE_MATERIALS_PATH` to the folder in your Drive that has your PDFs / DOCX / PPTX files




In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ============================================================
# CHANGE THIS to your folder path inside Google Drive
# ============================================================
COURSE_MATERIALS_PATH = "/content/drive/MyDrive/BMCB_658_Fa2025"  # <-- EDIT THIS

import os
if os.path.isdir(COURSE_MATERIALS_PATH):
    files = os.listdir(COURSE_MATERIALS_PATH)
    print(f"Found {len(files)} items in your course materials folder:")
    for f in files[:20]:
        print(f"  - {f}")
    if len(files) > 20:
        print(f"  ... and {len(files) - 20} more")
else:
    print(f"Folder not found: {COURSE_MATERIALS_PATH}")
    print("Please fix the path above and re-run this cell.")

Mounted at /content/drive
Found 8 items in your course materials folder:
  - Rag_Pepiline.ipynb
  - BMCB658_RAG_Optimized.ipynb
  - File
  - Publisher's ppt
  - Exams
  - Lecture Fa 2025
  - Recitation
  - chroma_db


## Step 4 Configuration

All settings in one place shared by the chat system **and** the evaluation suite.

In [ ]:
# ====================== CONFIGURATION ======================

OLLAMA_BASE_URL = "http://localhost:11434"
LLM_MODEL       = "llava:7b"          # Use "llava:7b" if you have Colab Pro
EMBED_MODEL     = "nomic-embed-text"

CHROMA_PERSIST_DIR = "/content/drive/MyDrive/chroma_db"
CHROMA_COLLECTION  = "biochem_course"

CHUNK_SIZE    = 800
CHUNK_OVERLAP = 100
RETRIEVER_K   = 5

IMAGES_DIR = "/content/drive/MyDrive/course_images"   # rendered slide/page images stored here

# Evaluation scoring rubric (used by LLM-as-judge in Steps 8-13) â”€â”€
SCORE_RUBRIC = {
    "faithfulness": (
        "0 = answer contradicts / ignores context; "
        "0.5 = partly grounded; "
        "1 = fully grounded in context"
    ),
    "relevance": (
        "0 = retrieved chunks are off-topic; "
        "0.5 = partly relevant; "
        "1 = highly relevant to the question"
    ),
    "compliance": (
        "0 = answer is vague/incorrect/dishonest; "
        "0.5 = acceptable but incomplete; "
        "1 = educational, accurate, honest, follows guidelines"
    ),
}

print("Configuration loaded.")
print(f"  LLM model:    {LLM_MODEL}")
print(f"  Embed model:  {EMBED_MODEL}")
print(f"  Course path:  {COURSE_MATERIALS_PATH}")
print(f"  Chroma dir:   {CHROMA_PERSIST_DIR}")
print(f"  Images dir:   {IMAGES_DIR}")

Configuration loaded.
  LLM model:    llava:7b
  Embed model:  nomic-embed-text
  Course path:  /content/drive/MyDrive/BMCB_658_Fa2025
  Chroma dir:   /content/drive/MyDrive/chroma_db
  Images dir:   /content/drive/MyDrive/course_images


## Step 5 Load, split, and index your course materials

This reads all your PDFs / DOCX / PPTX files, chops them into small chunks, and stores them in ChromaDB.

> **Run this only once** (or when you add new files to your Drive folder).  
> ChromaDB and the rendered images are saved to Google Drive and **persist across Colab sessions**.  
> On subsequent sessions, skip straight to Step 6 the index is already there.

In [ ]:
import os, subprocess
from pathlib import Path
from typing import List, Dict, Tuple

import fitz  # PyMuPDF
from langchain_core.documents import Document
from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    TextLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

# Document loaders
# so that text AND images are aligned by page/slide number.

TEXT_LOADERS = {
    ".pdf":  PyPDFLoader,
    ".docx": Docx2txtLoader,
    ".txt":  TextLoader,
    ".md":   TextLoader,
}

def _convert_pptx_to_pdf(pptx_path: str) -> str | None:
    """Convert a PPTX to PDF via LibreOffice. Returns the PDF path or None."""
    tmp_dir = "/tmp/pptx_convert"
    os.makedirs(tmp_dir, exist_ok=True)
    result = subprocess.run(
        ["libreoffice", "--headless", "--convert-to", "pdf", "--outdir", tmp_dir, pptx_path],
        capture_output=True, timeout=120,
    )
    pdf_path = os.path.join(tmp_dir, Path(pptx_path).stem + ".pdf")
    if os.path.exists(pdf_path):
        return pdf_path
    return None

def load_single_file(file_path: str) -> List[Document]:
    ext = Path(file_path).suffix.lower()
    source_name = Path(file_path).name

    if ext == ".pptx":
        pdf_path = _convert_pptx_to_pdf(file_path)
        if pdf_path is None:
            print(f"  [skip] Could not convert {source_name} to PDF")
            return []
        docs = PyPDFLoader(pdf_path).load()
        for doc in docs:
            doc.metadata["source"] = source_name
            doc.metadata["full_path"] = file_path
        return docs

    loader_cls = TEXT_LOADERS.get(ext)
    if loader_cls is None:
        return []
    try:
        docs = loader_cls(file_path).load()
        for doc in docs:
            doc.metadata["source"] = source_name
            doc.metadata["full_path"] = file_path
        return docs
    except Exception as exc:
        print(f"  [skip] Could not load {source_name}: {exc}")
        return []

def load_directory(directory: str) -> List[Document]:
    all_docs = []
    for root, _, files in os.walk(directory):
        for fname in sorted(files):
            fpath = os.path.join(root, fname)
            loaded = load_single_file(fpath)
            if loaded:
                print(f"  Loaded {fname} ({len(loaded)} pages/sections)")
            all_docs.extend(loaded)
    print(f"\nTotal: {len(all_docs)} document sections")
    return all_docs

# Image rendering (PDF pages / PPTX slides PNG)
def _render_pdf_pages(pdf_path: str, source_name: str, output_dir: str,
                      image_map: Dict[Tuple[str, int], str], dpi: int = 150):
    """Render each page of a PDF as a PNG image."""
    doc = fitz.open(pdf_path)
    for page_num in range(len(doc)):
        page = doc[page_num]
        pix = page.get_pixmap(dpi=dpi)
        stem = Path(source_name).stem
        img_name = f"{stem}_p{page_num}.png"
        img_path = os.path.join(output_dir, img_name)
        pix.save(img_path)
        image_map[(source_name, page_num)] = img_path
    doc.close()

def render_source_images(directory: str, output_dir: str) -> Dict[Tuple[str, int], str]:
    """Render all PDF pages and PPTX slides as images.
    Returns dict mapping (source_filename, page_number) -> image_path.
    """
    os.makedirs(output_dir, exist_ok=True)
    image_map: Dict[Tuple[str, int], str] = {}

    for root, _, files in os.walk(directory):
        for fname in sorted(files):
            fpath = os.path.join(root, fname)
            ext = Path(fname).suffix.lower()

            if ext == ".pdf":
                print(f"  Rendering pages: {fname}")
                _render_pdf_pages(fpath, fname, output_dir, image_map)

            elif ext == ".pptx":
                print(f"  Rendering slides: {fname}")
                pdf_path = _convert_pptx_to_pdf(fpath)
                if pdf_path:
                    _render_pdf_pages(pdf_path, fname, output_dir, image_map)

    print(f"\nRendered {len(image_map)} page/slide images to {output_dir}")
    return image_map

# Splitter, embeddings, vector store

def get_text_splitter():
    return RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

def get_embeddings():
    return OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_BASE_URL)

def get_vectorstore():
    return Chroma(
        collection_name=CHROMA_COLLECTION,
        persist_directory=CHROMA_PERSIST_DIR,
        embedding_function=get_embeddings(),
    )

def ingest_documents(docs: List[Document],
                     image_map: Dict[Tuple[str, int], str] | None = None) -> Chroma:
    splitter = get_text_splitter()
    chunks = splitter.split_documents(docs)

    if image_map:
        for chunk in chunks:
            source = chunk.metadata.get("source", "")
            page = chunk.metadata.get("page")
            if page is not None:
                key = (source, int(page))
                if key in image_map:
                    chunk.metadata["image_file"] = Path(image_map[key]).name

    print(f"Split into {len(chunks)} chunks")
    vs = get_vectorstore()
    if chunks:
        vs.add_documents(chunks)
        print(f"Stored {len(chunks)} chunks in ChromaDB")
    return vs

#  Shared retrieval helper (used by both chat agent and evaluator)

def retrieve_with_docs(query: str) -> Tuple[str, List[Document]]:
    """Search course materials and return formatted text + raw docs."""
    retriever = get_vectorstore().as_retriever(search_kwargs={"k": RETRIEVER_K})
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant course material found for this query.", []

    parts = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "")
        header = f"[Source {i}: {source}"
        if page != "":
            header += f", p.{page}"
        header += "]"
        parts.append(f"{header}\n{doc.page_content}")

    return "\n\n---\n\n".join(parts), docs

print("Ingestion + image rendering functions ready.")

Ingestion + image rendering functions ready.


In [ ]:
#  Run the ingestion + image rendering
# Skipped automatically if ChromaDB already contains data
# Set FORCE_REINDEX = True only if you added/changed course files.

FORCE_REINDEX = False   # <-- change to True to re-ingest from scratch

vs = get_vectorstore()
existing_count = vs._collection.count()

if existing_count > 0 and not FORCE_REINDEX:
    print(f"ChromaDB already has {existing_count} chunks â€” skipping ingestion.")
    print("Set FORCE_REINDEX = True above if you have added new course files.")

    # Rebuild image_map from the images already on Drive (needed by the chat UI)
    image_map = {}
    if os.path.isdir(IMAGES_DIR):
        for img_file in os.listdir(IMAGES_DIR):
            if img_file.endswith(".png"):
                # filename format: <stem>_p<page>.png
                img_path = os.path.join(IMAGES_DIR, img_file)
                try:
                    stem, page_part = img_file.rsplit("_p", 1)
                    page_num = int(page_part.replace(".png", ""))
                    # find a matching source name in the vectorstore metadata
                    image_map[(img_file, page_num)] = img_path
                except ValueError:
                    pass
        print(f"Found {len(image_map)} existing page images in {IMAGES_DIR}")
    else:
        print("Images directory not found on Drive â€” slide images won't show in chat.")

else:
    if FORCE_REINDEX:
        print("FORCE_REINDEX=True â€” clearing existing ChromaDB collection...")
        vs._collection.delete(where={"source": {"$ne": ""}})   # clear all docs
        print("Collection cleared.")

    print("=== Loading course documents ===")
    docs = load_directory(COURSE_MATERIALS_PATH)

    if docs:
        print("\n=== Rendering slide/page images ===")
        image_map = render_source_images(COURSE_MATERIALS_PATH, IMAGES_DIR)

        print("\n=== Embedding and indexing ===")
        vs = ingest_documents(docs, image_map=image_map)
        count = vs._collection.count()
        print(f"\nDone! ChromaDB has {count} searchable chunks, {len(image_map)} page images rendered.")
    else:
        print("No documents found. Check your COURSE_MATERIALS_PATH above.")
        image_map = {}

ChromaDB already has 8922 chunks â€” skipping ingestion.
Set FORCE_REINDEX = True above if you have added new course files.
Found 4490 existing page images in /content/drive/MyDrive/course_images


## Step 6 Build the agentic RAG agent

This cell defines:
1. **Prompts**  instructions for the AI (system prompt, router, grader, answer generator, rewriter)
2. **LangGraph agent** the decision-making graph that routes between chat and RAG

In [ ]:
from typing import Literal, TypedDict, Annotated
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END

# =====================================================================
# 1. PROMPTS
# =====================================================================

SYSTEM_PROMPT = (
    "You are a knowledgeable and friendly General Biochemistry Study Buddy  . "
    "Your role is to help students understand biochemistry concepts from their "
    "course materials. When answering:\n"
    "- Be accurate and pedagogical; explain concepts step by step.\n"
    "- Prefer information from the course materials over general knowledge.\n"
    "- If you are unsure, say so honestly rather than guessing.\n"
    "- Use clear examples, analogies, and diagram descriptions when helpful.\n"
    "- If the student uploads an image (e.g. a reaction mechanism or molecular "
    "structure), describe and analyze it."
)

ROUTER_PROMPT = (
    "You are a query classifier for a General Biochemistry Study Buddy  . "
    "Given the student's message, decide whether the question requires "
    "searching the course materials or can be answered with general chat.\n\n"
    "Respond with EXACTLY one word:\n"
    "- RETRIEVE  if the question is about specific biochemistry course content "
    "(enzymes, pathways, protein structure, nucleic acids, bioenergetics, "
    "specific lecture topics, homework problems, etc.)\n"
    "- CHAT  if the question is a greeting, general conversation, or can be "
    "answered without course-specific information.\n\n"
    "Student message: {question}\n\nDecision:"
)

GRADE_PROMPT = (
    "You are a grader assessing whether a retrieved document is relevant to "
    "a student's biochemistry question.\n\n"
    "Retrieved document:\n{context}\n\n"
    "Student question: {question}\n\n"
    "If the document contains keywords or semantic meaning related to the "
    "question, it is relevant. Respond with EXACTLY one word: YES or NO."
)

GENERATE_PROMPT = (
    "You are a General Biochemistry Study Buddy  . Use the following pieces of "
    "course material to answer the student's question. If the material does not "
    "fully answer the question, supplement with your knowledge but clearly "
    "indicate which parts come from course materials and which are general.\n\n"
    "Course material:\n{context}\n\n"
    "Student question: {question}\n\nProvide a clear, educational answer:"
)

REWRITE_PROMPT = (
    "The initial search did not return relevant course materials. "
    "Rewrite the following student question to improve retrieval. "
    "Focus on the core biochemistry concepts and use precise terminology.\n\n"
    "Original question: {question}\n\nRewritten question:"
)

# =====================================================================
# 2. LANGGRAPH AGENT
# =====================================================================

def _merge_messages(left: list, right: list) -> list:
    return left + right

def _merge_docs(left: list, right: list) -> list:
    return right if right else left

class AgentState(TypedDict):
    messages:        Annotated[list, _merge_messages]
    route:           str
    retrieved_docs:  Annotated[List[Document], _merge_docs]
    retrieved_text:  str
    retrieval_query: str
    grade_result:    str
    rewrite_count:   int

MAX_REWRITES = 2 # `MAX_REWRITES` | 2 | Allows self-correction without infinite loops on unanswerable questions |

def _llm():
    return ChatOllama(model=LLM_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.3) # `temperature` | 0.3 | Slightly creative for natural answers, low enough for consistency in evaluation |

# --- Nodes ---

def route_query(state: AgentState) -> dict:
    llm = _llm()
    user_msg = ""
    for msg in reversed(state["messages"]):
        if isinstance(msg, HumanMessage):
            user_msg = msg.content if isinstance(msg.content, str) else " ".join(
                p.get("text", "") for p in msg.content if isinstance(p, dict) and p.get("type") == "text"
            )
            break
    prompt = ROUTER_PROMPT.format(question=user_msg)
    response = llm.invoke([HumanMessage(content=prompt)])
    decision = response.content.strip().upper()
    route = "retrieve" if "RETRIEVE" in decision else "chat"
    return {"route": route, "retrieval_query": user_msg}

def retrieve(state: AgentState) -> dict:
    query = state.get("retrieval_query", "")
    text, docs = retrieve_with_docs(query)
    return {"retrieved_text": text, "retrieved_docs": docs}

def grade_documents(state: AgentState) -> dict:
    llm = _llm()
    question = state.get("retrieval_query", "")
    context = state.get("retrieved_text", "")
    if not context or context.startswith("No relevant"):
        return {"grade_result": "not_relevant"}
    prompt = GRADE_PROMPT.format(context=context[:3000], question=question)
    response = llm.invoke([HumanMessage(content=prompt)])
    answer = response.content.strip().upper()
    return {"grade_result": "relevant" if "YES" in answer else "not_relevant"}

def rewrite_question(state: AgentState) -> dict:
    llm = _llm()
    question = state.get("retrieval_query", "")
    prompt = REWRITE_PROMPT.format(question=question)
    response = llm.invoke([HumanMessage(content=prompt)])
    new_query = response.content.strip()
    return {"retrieval_query": new_query, "rewrite_count": state.get("rewrite_count", 0) + 1}

def generate_answer(state: AgentState) -> dict:
    llm = _llm()
    route = state.get("route", "chat")
    user_msg = ""
    original_human = None
    for msg in reversed(state["messages"]):
        if isinstance(msg, HumanMessage):
            original_human = msg
            user_msg = msg.content if isinstance(msg.content, str) else " ".join(
                p.get("text", "") for p in msg.content if isinstance(p, dict) and p.get("type") == "text"
            )
            break

    if route == "retrieve" and state.get("retrieved_text"):
        prompt_text = GENERATE_PROMPT.format(context=state["retrieved_text"], question=user_msg)
    else:
        prompt_text = user_msg

    sys_msg = SystemMessage(content=SYSTEM_PROMPT)
    if original_human and not isinstance(original_human.content, str):
        image_parts = [p for p in original_human.content if isinstance(p, dict) and p.get("type") == "image_url"]
        content_parts = [{"type": "text", "text": prompt_text}] + image_parts
        human = HumanMessage(content=content_parts)
    else:
        human = HumanMessage(content=prompt_text)

    response = llm.invoke([sys_msg, human])
    return {"messages": [AIMessage(content=response.content)]}

# --- Conditional edges ---

def after_route(state: AgentState) -> Literal["retrieve", "generate_answer"]:
    return "retrieve" if state.get("route") == "retrieve" else "generate_answer"

def after_grade(state: AgentState) -> Literal["generate_answer", "rewrite_question"]:
    if state.get("grade_result") == "relevant":
        return "generate_answer"
    if state.get("rewrite_count", 0) >= MAX_REWRITES:
        return "generate_answer"
    return "rewrite_question"

# --- Build the graph ---

def build_graph():
    workflow = StateGraph(AgentState)
    workflow.add_node("route_query", route_query)
    workflow.add_node("retrieve", retrieve)
    workflow.add_node("grade_documents", grade_documents)
    workflow.add_node("rewrite_question", rewrite_question)
    workflow.add_node("generate_answer", generate_answer)

    workflow.add_edge(START, "route_query")
    workflow.add_conditional_edges("route_query", after_route, {
        "retrieve": "retrieve",
        "generate_answer": "generate_answer",
    })
    workflow.add_edge("retrieve", "grade_documents")
    workflow.add_conditional_edges("grade_documents", after_grade, {
        "generate_answer": "generate_answer",
        "rewrite_question": "rewrite_question",
    })
    workflow.add_edge("rewrite_question", "retrieve")
    workflow.add_edge("generate_answer", END)
    return workflow.compile()

graph = build_graph()
print("Agent graph built successfully!")

Agent graph built successfully!


## Step 8 Test question set

Edit or extend `EVAL_QUESTIONS` with questions relevant to your course.  
Providing a `reference_answer` is optional but helps interpret faithfulness scores.

In [ ]:
EVAL_QUESTIONS = [
    {
        "question": "What is the role of ATP synthase in oxidative phosphorylation?",
        "reference_answer": "ATP synthase uses the proton gradient across the inner mitochondrial membrane to drive synthesis of ATP from ADP and inorganic phosphate."
    },
    {
        "question": "Explain the difference between competitive and non-competitive enzyme inhibition.",
        "reference_answer": "Competitive inhibitors bind the active site and increase apparent Km; non-competitive inhibitors bind an allosteric site, decrease Vmax, and do not change Km."
    },
    {
        "question": "What are the key steps of the citric acid cycle?",
        "reference_answer": "The TCA cycle starts with acetyl-CoA condensing with oxaloacetate to form citrate, proceeds through isocitrate, alpha-ketoglutarate, succinyl-CoA, succinate, fumarate, malate, and back to oxaloacetate, generating NADH, FADH2, GTP, and CO2."
    },
    {
        "question": "How do chaperone proteins assist in protein folding?",
        "reference_answer": "Chaperones (e.g., Hsp70, GroEL) prevent aggregation of unfolded polypeptides by transiently binding hydrophobic regions and providing a protected environment for correct folding."
    },
    {
        "question": "What is the central dogma of molecular biology?",
        "reference_answer": "DNA is transcribed into RNA, which is translated into protein. Information flows from DNA to RNA to protein."
    },
]

print(f"{len(EVAL_QUESTIONS)} evaluation questions loaded.")

5 evaluation questions loaded.


## Step 9 — Embedding-based scoring functions

All three metrics use **cosine similarity** between `sentence-transformers` embeddings
| Metric | Embedding comparison |
|---|---|
| **Faithfulness** | cosine_sim(answer, retrieved context) — is the answer grounded in what was found? |
| **Relevance** | cosine_sim(question, retrieved context) — are the chunks on-topic? |
| **Compliance** | cosine_sim(answer, question) — does the answer address the question? |



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load once — small model (~80 MB), runs on CPU, no GPU contention with Ollama
_embedder = SentenceTransformer("all-MiniLM-L6-v2")

def _embed(texts: list) -> np.ndarray:
    return _embedder.encode(texts, normalize_embeddings=True)

def score_all(question: str, context: str, answer: str) -> dict:
    """Return Faithfulness, Relevance, Compliance as cosine similarities."""
    vecs = _embed([question, context[:2000], answer])
    q_vec, c_vec, a_vec = vecs[0:1], vecs[1:2], vecs[2:3]
    return {
        "faithfulness": round(float(cosine_similarity(a_vec, c_vec)[0][0]), 3),
        "relevance":    round(float(cosine_similarity(q_vec, c_vec)[0][0]), 3),
        "compliance":   round(float(cosine_similarity(a_vec, q_vec)[0][0]), 3),
    }

# ── Streamlined eval runner (skips router + grader) ───────────
# For evaluation we always want retrieval, so we bypass the routing
# and grading nodes and go straight to: retrieve → generate.

def run_agent_eval(question: str):
    """Retrieve context then generate an answer — no router or grader.
    Returns (answer, retrieved_context_text, retrieved_docs).
    """
    # Step 1: retrieve
    context_text, retrieved_docs = retrieve_with_docs(question)

    # Step 2: generate
    prompt_text = GENERATE_PROMPT.format(context=context_text, question=question)
    llm = ChatOllama(model=LLM_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.3)
    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=prompt_text),
    ])
    return response.content.strip(), context_text, retrieved_docs

print("Embedding scorer and streamlined eval runner ready.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding scorer and streamlined eval runner ready.


## Step 7  Launch the Chat UI

Run this cell to start the **Gradio** chat interface. It will:
- Show a **chat window** where you type your questions
- Show a **Trace** panel with the steps the agent took (routing, searching, grading, answering)
- Show a **Sources** panel with the exact course material chunks used
- Show which **route** was taken (course materials vs general knowledge)

You'll get a **public link** you can share with classmates!

In [ ]:
import gradio as gr

NODE_LABELS = {
    "route_query":      "Deciding whether to search course materials",
    "retrieve":         "Searching course materials",
    "grade_documents":  "Checking relevance of results",
    "rewrite_question": "Refining the search query",
    "generate_answer":  "Writing answer",
}

chat_history_messages = []


def _collect_images(retrieved_docs) -> list[str]:
    """Get unique image file paths from retrieved chunks."""
    seen = set()
    paths = []
    for doc in retrieved_docs:
        img_file = doc.metadata.get("image_file", "")
        if img_file and img_file not in seen:
            img_path = os.path.join(IMAGES_DIR, img_file)
            if os.path.exists(img_path):
                seen.add(img_file)
                source = doc.metadata.get("source", "")
                page = doc.metadata.get("page", "")
                caption = source
                if page != "":
                    caption += f", p.{page}"
                paths.append((img_path, caption))
    return paths


def run_agent(user_message: str, chat_history: list):
    """Process a user message through the agent graph and return results."""
    global chat_history_messages

    human_msg = HumanMessage(content=user_message)

    input_state = {
        "messages": chat_history_messages + [human_msg],
        "route": "",
        "retrieved_docs": [],
        "retrieved_text": "",
        "retrieval_query": "",
        "grade_result": "",
        "rewrite_count": 0,
    }

    trace_lines = []
    retrieved_docs = []
    route_taken = "chat"
    final_answer = ""

    for chunk in graph.stream(input_state):
        for node_name, node_output in chunk.items():
            label = NODE_LABELS.get(node_name, node_name)

            if node_name == "route_query":
                route_taken = node_output.get("route", "chat")
                trace_lines.append(f"**{label}** â†’ {route_taken.upper()}")

            elif node_name == "retrieve":
                retrieved_docs = node_output.get("retrieved_docs", [])
                trace_lines.append(f"**{label}** â†’ {len(retrieved_docs)} chunks found")

            elif node_name == "grade_documents":
                grade = node_output.get("grade_result", "")
                trace_lines.append(f"**{label}** â†’ {grade}")

            elif node_name == "rewrite_question":
                new_q = node_output.get("retrieval_query", "")
                trace_lines.append(f"**{label}** â†’ _{new_q}_")

            elif node_name == "generate_answer":
                msgs = node_output.get("messages", [])
                if msgs:
                    final_answer = msgs[-1].content
                trace_lines.append(f"**{label}** â†’ done")

    # Sources text
    sources_text = ""
    if retrieved_docs:
        for i, doc in enumerate(retrieved_docs, 1):
            src = doc.metadata.get("source", "unknown")
            page = doc.metadata.get("page", "")
            header = f"### Source {i}: {src}"
            if page != "":
                header += f" (p.{page})"
            sources_text += f"{header}\n{doc.page_content}\n\n---\n\n"
    else:
        sources_text = "_No course materials were retrieved for this answer._"

    # Trace text
    trace_text = "\n\n".join(f"{i+1}. {line}" for i, line in enumerate(trace_lines))

    # Route badge
    if route_taken == "retrieve":
        route_info = "Answered from **course materials**"
    else:
        route_info = "Answered from **general knowledge**"

    # Collect slide/page images linked to retrieved chunks
    gallery_items = _collect_images(retrieved_docs)

    # Update conversation memory
    chat_history_messages.append(human_msg)
    chat_history_messages.append(AIMessage(content=final_answer))

    chat_history = chat_history or []
    chat_history.append((user_message, final_answer))

    return chat_history, trace_text, sources_text, route_info, gallery_items


def clear_all():
    """Reset the conversation."""
    global chat_history_messages
    chat_history_messages = []
    return [], "", "", "", []


# Build the Gradio interface

with gr.Blocks(
    title="Biochem TA",
    theme=gr.themes.Soft(),
) as demo:
    gr.Markdown("# General Biochemistry Study Buddy")
    gr.Markdown(
        "Ask questions about your biochemistry course. "
        "The AI will search your course materials and show relevant **slide/page images** alongside the answer."
    )

    with gr.Row():
        # Left column: chat
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Chat",
                height=500,
                show_copy_button=True,
            )
            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder="Ask a biochemistry question...",
                    show_label=False,
                    scale=5,
                    container=False,
                )
                send_btn = gr.Button("Send", variant="primary", scale=1)
            clear_btn = gr.Button("Clear conversation", variant="secondary")

        # Right column: info panels
        with gr.Column(scale=2):
            route_display = gr.Markdown(value="", label="Route")
            with gr.Accordion("Slide / Page Images", open=True):
                image_gallery = gr.Gallery(
                    label="Retrieved course images",
                    columns=1,
                    height=300,
                    object_fit="contain",
                )
            with gr.Accordion("Trace (agent steps)", open=False):
                trace_display = gr.Markdown(value="_Send a message to see the trace._")
            with gr.Accordion("Sources (retrieved text)", open=False):
                sources_display = gr.Markdown(value="_Send a message to see sources._")

    outputs = [chatbot, trace_display, sources_display, route_display, image_gallery]

    send_btn.click(
        fn=run_agent,
        inputs=[msg_input, chatbot],
        outputs=outputs,
    ).then(lambda: "", outputs=msg_input)

    msg_input.submit(
        fn=run_agent,
        inputs=[msg_input, chatbot],
        outputs=outputs,
    ).then(lambda: "", outputs=msg_input)

    clear_btn.click(
        fn=clear_all,
        outputs=outputs,
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_1435/1857191838.py:124: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_1435/1857191838.py:137: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_1435/1857191838.py:137: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_1435/1857191838.py:137: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to d

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2bfe377feea54ac663.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Step 10 — Run evaluation

For each test question this cell makes exactly **1 LLM call** (retrieve + generate).
Scoring is done with `sentence-transformers` cosine similarity and takes milliseconds.

Expected runtime: **~30–60 s per question** on a T4 GPU → ~2–4 min for 5 questions.


In [ ]:
import time

eval_results = []

for i, item in enumerate(EVAL_QUESTIONS, 1):
    question = item["question"]
    t0 = time.time()
    print(f"\n[{i}/{len(EVAL_QUESTIONS)}] {question[:80]}...")

    # 1 LLM call: retrieve + generate
    print("  Generating answer...")
    answer, context_text, retrieved_docs = run_agent_eval(question)

    # Embedding-based scoring — milliseconds, no LLM
    scores = score_all(question, context_text, answer)

    avg = round((scores['faithfulness'] + scores['relevance'] + scores['compliance']) / 3, 3)
    result = {
        "question":   question,
        "answer":     answer,
        "num_chunks": len(retrieved_docs),
        **scores,
        "avg_score":  avg,
    }
    eval_results.append(result)
    elapsed = round(time.time() - t0, 1)
    print(f"  Faith={scores['faithfulness']:.3f}  Rel={scores['relevance']:.3f}  "
          f"Comp={scores['compliance']:.3f}  Avg={avg:.3f}  ({elapsed}s)")

eval_df = pd.DataFrame(eval_results)
print(f"\n=== Evaluation complete — {len(eval_results)} questions ===")


[1/5] What is the role of ATP synthase in oxidative phosphorylation?...
  Generating answer...
  Faith=0.705  Rel=0.563  Comp=0.822  Avg=0.697  (741.7s)

[2/5] Explain the difference between competitive and non-competitive enzyme inhibition...
  Generating answer...
  Faith=0.791  Rel=0.705  Comp=0.875  Avg=0.790  (713.9s)

[3/5] What are the key steps of the citric acid cycle?...
  Generating answer...
  Faith=0.671  Rel=0.599  Comp=0.682  Avg=0.651  (859.0s)

[4/5] How do chaperone proteins assist in protein folding?...
  Generating answer...
  Faith=0.898  Rel=0.715  Comp=0.824  Avg=0.812  (804.8s)

[5/5] What is the central dogma of molecular biology?...
  Generating answer...
  Faith=0.447  Rel=0.336  Comp=0.688  Avg=0.490  (673.4s)

=== Evaluation complete — 5 questions ===


## Step 11 Results table

In [ ]:
display_cols = ["question", "num_chunks", "faithfulness", "relevance", "compliance", "avg_score"]

styled = (
    eval_df[display_cols]
    .style
    .background_gradient(
        subset=["faithfulness", "relevance", "compliance", "avg_score"],
        cmap="RdYlGn", vmin=0, vmax=1
    )
    .format({"faithfulness": "{:.3f}", "relevance": "{:.3f}",
             "compliance": "{:.3f}", "avg_score": "{:.3f}"})
    .set_properties(**{"white-space": "pre-wrap", "max-width": "400px"})
)
display(styled)

print("\n=== Aggregate Scores ===")
print(f"  Mean Faithfulness : {eval_df['faithfulness'].mean():.3f}")
print(f"  Mean Relevance    : {eval_df['relevance'].mean():.3f}")
print(f"  Mean Compliance   : {eval_df['compliance'].mean():.3f}")
print(f"  Overall Average   : {eval_df['avg_score'].mean():.3f}")

,question,num_chunks,faithfulness,relevance,compliance,avg_score
0,What is the role of ATP synthase in oxidative phosphorylation?,5,0.705,0.563,0.822,0.697
1,Explain the difference between competitive and non-competitive enzyme inhibition.,5,0.791,0.705,0.875,0.790
2,What are the key steps of the citric acid cycle?,5,0.671,0.599,0.682,0.651
3,How do chaperone proteins assist in protein folding?,5,0.898,0.715,0.824,0.812
4,What is the central dogma of molecular biology?,5,0.447,0.336,0.688,0.490



=== Aggregate Scores ===
  Mean Faithfulness : 0.702
  Mean Relevance    : 0.584
  Mean Compliance   : 0.778
  Overall Average   : 0.688


## Step 12 Visualisation

In [ ]:
metrics = ["faithfulness", "relevance", "compliance"]
means   = [eval_df[m].mean() for m in metrics]
colors  = ["#4C72B0", "#55A868", "#C44E52"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Agentic RAG Evaluation â€” Biochemistry Study Buddy", fontsize=14, fontweight="bold")

#  Left: aggregate bar chart
ax1 = axes[0]
bars = ax1.bar(metrics, means, color=colors, edgecolor="white", linewidth=1.5)
ax1.set_ylim(0, 1.05)
ax1.set_ylabel("Mean Score (0 â€“ 1)")
ax1.set_title("Aggregate Metric Scores")
ax1.axhline(0.7, color="grey", linestyle="--", linewidth=0.8, label="0.7 threshold")
ax1.legend(fontsize=9)
for bar, val in zip(bars, means):
    ax1.text(bar.get_x() + bar.get_width() / 2, val + 0.02, f"{val:.3f}",
             ha="center", va="bottom", fontsize=11, fontweight="bold")

# â Right: per-question grouped bar chart
ax2 = axes[1]
n_q   = len(eval_df)
x     = np.arange(n_q)
width = 0.25
for j, (metric, color) in enumerate(zip(metrics, colors)):
    ax2.bar(x + j * width, eval_df[metric], width, label=metric.capitalize(), color=color, alpha=0.85)
ax2.set_xticks(x + width)
ax2.set_xticklabels([f"Q{i+1}" for i in range(n_q)], fontsize=10)
ax2.set_ylim(0, 1.1)
ax2.set_ylabel("Score")
ax2.set_title("Per-Question Scores")
ax2.axhline(0.7, color="grey", linestyle="--", linewidth=0.8)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig("/content/rag_eval_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to /content/rag_eval_results.png")

## Step 13 — Detailed answer log

Inspect the scores and generated answers for every question.


In [ ]:
for i, row in eval_df.iterrows():
    print(f"
{'='*70}")
    print(f"Q{i+1}: {row['question']}")
    print(f"  Chunks retrieved : {row['num_chunks']}")
    print(f"  Faithfulness : {row['faithfulness']:.3f}")
    print(f"  Relevance    : {row['relevance']:.3f}")
    print(f"  Compliance   : {row['compliance']:.3f}")
    print(f"  Avg Score    : {row['avg_score']:.3f}")
    print(f"
  Answer preview: {row['answer'][:300]}...")


## Step 14 Save results to CSV

In [ ]:
from google.colab import files as colab_files

csv_path = "/content/rag_eval_results.csv"
eval_df.to_csv(csv_path, index=False)
print(f"Results saved to {csv_path}")

colab_files.download(csv_path)
colab_files.download("/content/rag_eval_results.png")